# Research Agents Example Notebook

This notebook demonstrates how to use the research agents for financial data analysis.

The agents module provides:
- **DataCollectionAgent**: Collect financial market data
- **StatisticalAnalysisAgent**: Perform statistical analysis
- **RiskAnalyticsAgent**: Analyze portfolio risk
- **MarketResearchAgent**: Market research and trends
- **BacktestingAgent**: Backtest trading strategies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')

from src.agents import (
    DataCollectionAgent,
    StatisticalAnalysisAgent,
    RiskAnalyticsAgent,
    MarketResearchAgent,
    BacktestingAgent
)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Collection Agent

Let's start by collecting data for a portfolio of ETFs.

In [ ]:
# Initialize the data collection agent
data_agent = DataCollectionAgent()

# Define our universe of assets
tickers = ['SPY', 'TLT', 'GLD', 'IEF', 'VNQ']

# Download 5 years of daily data
prices = data_agent.execute(
    tickers=tickers,
    period='5y',
    interval='1d'
)

# Extract adjusted close prices
prices_adj = prices['Adj Close']
print(f"Downloaded {len(prices_adj)} days of data")
prices_adj.tail()

In [ ]:
# Validate data quality
validation = data_agent.validate_data(prices_adj)
print("Data Validation Report:")
print(f"Total rows: {validation['total_rows']}")
print(f"Date range: {validation['date_range']['start']} to {validation['date_range']['end']}")
print(f"Missing values: {validation['missing_values']}")
print(f"Duplicates: {validation['duplicates']}")

In [ ]:
# Calculate returns
returns = data_agent.calculate_returns(prices_adj, method='simple')
returns = returns.dropna()

# Plot normalized prices
normalized_prices = (prices_adj / prices_adj.iloc[0]) * 100
normalized_prices.plot(title='Normalized Prices (Base = 100)', figsize=(12, 6))
plt.ylabel('Normalized Price')
plt.legend(loc='best')
plt.show()

## 2. Statistical Analysis Agent

Perform comprehensive statistical analysis on our returns.

In [ ]:
# Initialize the statistical analysis agent
stats_agent = StatisticalAnalysisAgent()

# Descriptive statistics
desc_stats = stats_agent.descriptive_statistics(returns)
print("\nBasic Statistics:")
print(desc_stats['basic_stats'])
print("\nSkewness:")
print(desc_stats['skewness'])
print("\nKurtosis:")
print(desc_stats['kurtosis'])

In [ ]:
# Correlation analysis
corr_analysis = stats_agent.correlation_analysis(returns, method='pearson')

# Plot correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(corr_analysis['correlation'], annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.show()

# Covariance matrix
print("\nCovariance Matrix:")
print(corr_analysis['covariance'])

In [ ]:
# Distribution analysis
dist_analysis = stats_agent.distribution_analysis(returns, test_normal=True)

# Display normality test results
print("Normality Tests:")
for ticker, results in dist_analysis.items():
    print(f"\n{ticker}:")
    if 'jarque_bera_test' in results:
        jb = results['jarque_bera_test']
        print(f"  Jarque-Bera: p-value = {jb['p_value']:.4f}, Normal = {jb['is_normal']}")
    print(f"  Skewness: {results['skewness']:.4f}")
    print(f"  Kurtosis: {results['kurtosis']:.4f}")

In [ ]:
# Outlier detection
outliers = stats_agent.outlier_detection(returns, method='iqr', threshold=1.5)

print("Outlier Detection Results (IQR method):")
for ticker, result in outliers.items():
    print(f"{ticker}: {result['outlier_count']} outliers ({result['outlier_pct']:.2f}%)")

## 3. Risk Analytics Agent

Analyze portfolio risk using various metrics.

In [ ]:
# Initialize the risk analytics agent
risk_agent = RiskAnalyticsAgent()

# Comprehensive risk analysis
risk_analysis = risk_agent.comprehensive_risk_analysis(
    returns,
    confidence_level=0.95,
    risk_free_rate=0.02
)

# Display results
print("Risk Analysis Results:\n")
for category, metrics in risk_analysis.items():
    print(f"\n{category.upper().replace('_', ' ')}:")
    if isinstance(metrics, dict):
        for metric, value in metrics.items():
            if isinstance(value, (float, np.floating)):
                print(f"  {metric}: {value:.4f}")
            else:
                print(f"  {metric}: {value}")
    else:
        print(f"  {metrics}")

In [ ]:
# Calculate VaR using different methods
var_methods = ['historical', 'parametric', 'cornish_fisher']
var_results = {}

for method in var_methods:
    var_results[method] = risk_agent.value_at_risk(
        returns,
        confidence_level=0.95,
        method=method
    )

# Create comparison DataFrame
var_df = pd.DataFrame(var_results)
print("\nValue at Risk (95% confidence) - Method Comparison:")
print(var_df)

# Plot VaR comparison
var_df.plot(kind='bar', title='VaR Comparison by Method')
plt.ylabel('VaR')
plt.xlabel('Asset')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate drawdowns
max_dd = risk_agent.maximum_drawdown(returns)

print("Maximum Drawdown:")
print(max_dd)

# Plot drawdown over time for SPY
cumulative = (1 + returns['SPY']).cumprod()
running_max = cumulative.expanding().max()
drawdown = (cumulative - running_max) / running_max

plt.figure(figsize=(12, 6))
drawdown.plot(title='SPY Drawdown Over Time')
plt.ylabel('Drawdown')
plt.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
plt.fill_between(drawdown.index, drawdown.values, 0, alpha=0.3)
plt.show()

## 4. Market Research Agent

Perform market research and technical analysis.

In [ ]:
# Initialize the market research agent
market_agent = MarketResearchAgent()

# Trend analysis
trends = market_agent.trend_analysis(prices_adj, windows=[20, 50, 200])

# Plot price with moving averages for SPY
plt.figure(figsize=(12, 6))
prices_adj['SPY'].plot(label='SPY Price', linewidth=2)
trends['SMA_20']['SPY'].plot(label='SMA 20', alpha=0.7)
trends['SMA_50']['SPY'].plot(label='SMA 50', alpha=0.7)
trends['SMA_200']['SPY'].plot(label='SMA 200', alpha=0.7)
plt.title('SPY Price with Moving Averages')
plt.legend()
plt.show()

In [ ]:
# Momentum analysis
momentum = market_agent.momentum_analysis(prices_adj, period=14)

# Plot RSI for SPY
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Price
ax1.plot(prices_adj.index, prices_adj['SPY'])
ax1.set_title('SPY Price')
ax1.set_ylabel('Price')

# RSI
ax2.plot(momentum['RSI'].index, momentum['RSI']['SPY'])
ax2.axhline(y=70, color='r', linestyle='--', label='Overbought')
ax2.axhline(y=30, color='g', linestyle='--', label='Oversold')
ax2.set_title('SPY RSI (14)')
ax2.set_ylabel('RSI')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Regime detection
regimes = market_agent.regime_detection(returns, method='volatility', window=60)

print("Market Regime Analysis:")
print("\nVolatility by Asset:")
print(regimes['volatility'].tail())

# Plot volatility regime for SPY
plt.figure(figsize=(12, 6))
regimes['volatility']['SPY'].plot(title='SPY Rolling Volatility (60-day)')
plt.ylabel('Volatility')
plt.show()

In [ ]:
# Market summary
summary = market_agent.market_summary(prices_adj, returns)

print("Market Summary:")
print("\nLatest Prices:")
for ticker, price in summary['latest_prices'].items():
    print(f"{ticker}: ${price:.2f}")

print("\n1-Day Price Change:")
for ticker, change in summary['price_change_1d'].items():
    print(f"{ticker}: {change*100:.2f}%")

if summary['price_change_1m']:
    print("\n1-Month Price Change:")
    for ticker, change in summary['price_change_1m'].items():
        print(f"{ticker}: {change*100:.2f}%")

## 5. Backtesting Agent

Backtest a simple equal-weight portfolio strategy.

In [ ]:
# Initialize the backtesting agent
backtest_agent = BacktestingAgent()

# Create equal-weight strategy
equal_weights = pd.DataFrame(
    1 / len(tickers),
    index=prices_adj.index,
    columns=prices_adj.columns
)

# Run backtest
results = backtest_agent.execute(
    prices=prices_adj,
    strategy=equal_weights,
    initial_capital=100000,
    commission=0.001,
    slippage=0.0005,
    rebalance_frequency='monthly'
)

# Display report
report = backtest_agent.generate_report(results, "Equal Weight Portfolio")
print(report)

In [ ]:
# Plot portfolio value over time
plt.figure(figsize=(12, 6))
results['portfolio_value'].plot(title='Portfolio Value Over Time', linewidth=2)
plt.ylabel('Portfolio Value ($)')
plt.axhline(y=100000, color='r', linestyle='--', label='Initial Capital', alpha=0.5)
plt.legend()
plt.show()

# Plot returns distribution
plt.figure(figsize=(12, 6))
results['returns'].hist(bins=50, edgecolor='black')
plt.title('Distribution of Daily Returns')
plt.xlabel('Return')
plt.ylabel('Frequency')
plt.axvline(x=0, color='r', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# Compare with buy-and-hold SPY
spy_returns = returns['SPY']
spy_value = 100000 * (1 + spy_returns).cumprod()

plt.figure(figsize=(12, 6))
results['portfolio_value'].plot(label='Equal Weight Portfolio', linewidth=2)
spy_value.plot(label='Buy & Hold SPY', linewidth=2)
plt.title('Strategy Comparison')
plt.ylabel('Portfolio Value ($)')
plt.legend()
plt.show()

# Calculate SPY metrics for comparison
spy_metrics = backtest_agent.calculate_performance_metrics(
    spy_returns,
    spy_value
)

print("\nSPY Buy & Hold Metrics:")
print(f"Total Return: {spy_metrics['total_return']*100:.2f}%")
print(f"Annual Return: {spy_metrics['annual_return']*100:.2f}%")
print(f"Sharpe Ratio: {spy_metrics['sharpe_ratio']:.2f}")
print(f"Max Drawdown: {spy_metrics['max_drawdown']*100:.2f}%")

## 6. Monte Carlo Simulation

Run Monte Carlo simulation to understand potential future outcomes.

In [ ]:
# Run Monte Carlo simulation
mc_results = backtest_agent.monte_carlo_simulation(
    returns=results['returns'],
    num_simulations=1000,
    num_periods=252,  # 1 year
    initial_capital=results['final_value']
)

print("Monte Carlo Simulation Results (1 year):")
print(f"\nMean Final Value: ${mc_results['mean_final_value']:,.2f}")
print(f"\nPercentiles:")
for pct, value in mc_results['percentiles'].items():
    print(f"  {pct}: ${value:,.2f}")
print(f"\nProbability of Loss: {mc_results['probability_of_loss']*100:.2f}%")

In [ ]:
# Plot Monte Carlo simulation paths
plt.figure(figsize=(12, 6))

# Plot a sample of paths
sample_paths = mc_results['portfolio_paths'][:100]
for path in sample_paths:
    plt.plot(path, alpha=0.1, color='blue')

# Plot percentiles
percentiles_array = np.percentile(mc_results['portfolio_paths'], [5, 50, 95], axis=0)
plt.plot(percentiles_array[0], 'r--', label='5th Percentile', linewidth=2)
plt.plot(percentiles_array[1], 'g-', label='Median', linewidth=2)
plt.plot(percentiles_array[2], 'r--', label='95th Percentile', linewidth=2)

plt.title('Monte Carlo Simulation - Portfolio Paths (1 Year)')
plt.xlabel('Days')
plt.ylabel('Portfolio Value ($)')
plt.legend()
plt.show()

## 7. Comprehensive Portfolio Analysis

Combine all agents for a complete portfolio analysis.

In [ ]:
# Create a comprehensive analysis function
def comprehensive_portfolio_analysis(tickers, period='5y'):
    """Perform comprehensive analysis using all agents."""

    results = {}

    # 1. Data Collection
    print("1. Collecting data...")
    data_agent = DataCollectionAgent()
    prices = data_agent.execute(tickers, period=period)['Adj Close']
    returns = data_agent.calculate_returns(prices).dropna()
    results['prices'] = prices
    results['returns'] = returns

    # 2. Statistical Analysis
    print("2. Performing statistical analysis...")
    stats_agent = StatisticalAnalysisAgent()
    results['statistics'] = stats_agent.descriptive_statistics(returns)
    results['correlation'] = stats_agent.correlation_analysis(returns)

    # 3. Risk Analysis
    print("3. Analyzing risk...")
    risk_agent = RiskAnalyticsAgent()
    results['risk'] = risk_agent.comprehensive_risk_analysis(returns)

    # 4. Market Research
    print("4. Conducting market research...")
    market_agent = MarketResearchAgent()
    results['market'] = market_agent.market_summary(prices, returns)

    # 5. Backtest
    print("5. Running backtest...")
    backtest_agent = BacktestingAgent()
    weights = pd.DataFrame(1/len(tickers), index=prices.index, columns=prices.columns)
    results['backtest'] = backtest_agent.execute(prices, weights)

    print("\nAnalysis complete!")
    return results

# Run comprehensive analysis
portfolio_analysis = comprehensive_portfolio_analysis(tickers)

In [ ]:
# Display key findings
print("=" * 60)
print("COMPREHENSIVE PORTFOLIO ANALYSIS SUMMARY")
print("="  * 60)

print("\nPORTFOLIO COMPOSITION:")
print(f"Assets: {', '.join(tickers)}")
print(f"Strategy: Equal Weight")

print("\nPERFORMANCE METRICS:")
metrics = portfolio_analysis['backtest']['metrics']
print(f"Total Return: {metrics['total_return']*100:.2f}%")
print(f"Annual Return: {metrics['annual_return']*100:.2f}%")
print(f"Annual Volatility: {metrics['annual_volatility']*100:.2f}%")

print("\nRISK-ADJUSTED METRICS:")
print(f"Sharpe Ratio: {metrics['sharpe_ratio']:.2f}")
print(f"Sortino Ratio: {metrics['sortino_ratio']:.2f}")
print(f"Calmar Ratio: {metrics['calmar_ratio']:.2f}")

print("\nRISK METRICS:")
print(f"Maximum Drawdown: {metrics['max_drawdown']*100:.2f}%")
print(f"Win Rate: {metrics['win_rate']*100:.2f}%")

print("\nCORRELATION SUMMARY:")
avg_corr = portfolio_analysis['correlation']['correlation'].values[
    np.triu_indices_from(portfolio_analysis['correlation']['correlation'].values, k=1)
].mean()
print(f"Average Pairwise Correlation: {avg_corr:.3f}")

print("\n" + "="*60)

## Conclusion

This notebook demonstrated the key capabilities of the research agents:

1. **Data Collection**: Automated data gathering and validation
2. **Statistical Analysis**: Comprehensive statistical testing and analysis
3. **Risk Analytics**: Multi-dimensional risk assessment
4. **Market Research**: Technical analysis and regime detection
5. **Backtesting**: Strategy validation with realistic costs

These agents can be combined in various ways to support your quantitative research and portfolio management workflows.

For more information, see the agents README at `src/agents/README.md`.

In [ ]:
# pcrm-book - Next generation investment analysis.
# Copyright (C) 2025 Anton Vorobets.

# This program is free software: you can redistribute it and/or modify
# it under the terms of the GNU General Public License as published by
# the Free Software Foundation, either version 3 of the License, or
# (at your option) any later version.

# This program is distributed in the hope that it will be useful,
# but WITHOUT ANY WARRANTY; without even the implied warranty of
# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the
# GNU General Public License for more details.

# You should have received a copy of the GNU General Public License
# along with this program.  If not, see <https://www.gnu.org/licenses/>.